In [ ]:
!pip install pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder \
    .appName("Lab3_SparkSQL") \
    .getOrCreate()

In [ ]:
df = spark.read.option("sep", "\t").option("header", "false") \
    .csv("data/ngram.csv") \
    .toDF("ngram", "Year", "Count", "Pages", "Books")

# Cast columns to proper types
df = df.withColumn("Year", F.col("Year").cast("int")) \
    .withColumn("Count", F.col("Count").cast("int")) \
    .withColumn("Pages", F.col("Pages").cast("int")) \
    .withColumn("Books", F.col("Books").cast("int"))

df.show(5)

+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1985|    1|    1|    1|
|! $17.95|1987|    1|    1|    1|
|! $17.95|1990|    1|    1|    1|
|! $17.95|1991|    1|    1|    1|
|! $17.95|1992|    5|    5|    5|
+--------+----+-----+-----+-----+
only showing top 5 rows


In [ ]:
df.createOrReplaceTempView("ngram_table")

In [ ]:
# 3.1) Return all bigrams where Count > 5
#SQL
spark.sql("SELECT * FROM ngram_table WHERE Count > 5").show(10)
#API
df.filter(F.col("Count") > 5).show(10)

+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1997|    6|    5|    5|
|! $17.95|1999|   11|   10|   10|
|! $17.95|2000|   11|    9|    9|
|! $17.95|2004|   14|   14|   14|
|! $17.95|2005|   13|   13|   13|
|    ! 09|1899|    6|    6|    5|
|    ! 09|1916|    7|    7|    4|
|    ! 09|1936|    6|    6|    6|
|    ! 09|1997|    6|    5|    5|
|    ! 09|1999|   11|   10|   10|
+--------+----+-----+-----+-----+
only showing top 10 rows


In [ ]:
# -----------------------------------------------------------------------------
# 3.3) Bigrams with the highest Count for each year
# -----------------------------------------------------------------------------
# SQL:
spark.sql("""
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY Year ORDER BY Count DESC) as rn
        FROM ngram_table
    )
    SELECT ngram, Year, Count, Pages, Books FROM ranked WHERE rn = 1
""").show()
# API:
window = Window.partitionBy("Year").orderBy(F.col("Count").desc())
df.withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .show()

+-----+----+-----+-----+-----+
|ngram|Year|Count|Pages|Books|
+-----+----+-----+-----+-----+
| ! 09|1780|    1|    1|    1|
| ! 09|1803|    1|    1|    1|
| ! 09|1806|    1|    1|    1|
| ! 09|1823|    1|    1|    1|
| ! 09|1824|    1|    1|    1|
| ! 09|1825|    1|    1|    1|
| ! 09|1829|    3|    3|    3|
| ! 09|1830|    2|    2|    2|
| ! 09|1831|    1|    1|    1|
| ! 09|1833|    1|    1|    1|
| ! 09|1834|    4|    4|    4|
| ! 09|1836|    1|    1|    1|
| ! 09|1839|    1|    1|    1|
| ! 09|1840|    1|    1|    1|
| ! 09|1841|    2|    2|    2|
| ! 09|1845|    1|    1|    1|
| ! 09|1847|    2|    2|    2|
| ! 09|1848|    1|    1|    1|
| ! 09|1856|    1|    1|    1|
| ! 09|1857|    2|    2|    2|
+-----+----+-----+-----+-----+
only showing top 20 rows
+-----+----+-----+-----+-----+
|ngram|Year|Count|Pages|Books|
+-----+----+-----+-----+-----+
| ! 09|1780|    1|    1|    1|
| ! 09|1803|    1|    1|    1|
| ! 09|1806|    1|    1|    1|
| ! 09|1823|    1|    1|    1|
| ! 09|1824|  

In [ ]:
# 3.4) All bigrams that appeared in 20 different years
# -----------------------------------------------------------------------------
# SQL:
spark.sql("""
    SELECT ngram
    FROM ngram_table
    GROUP BY ngram
    HAVING COUNT(DISTINCT Year) = 20
""").show()
# API:
df.groupBy("ngram") \
    .agg(F.countDistinct("Year").alias("years_count")) \
    .filter(F.col("years_count") == 20) \
    .select("ngram") \
    .show()

+--------+
|   ngram|
+--------+
|! $17.95|
+--------+

+--------+
|   ngram|
+--------+
|! $17.95|
+--------+



In [ ]:
# 3.5) Bigrams with '!' in first part and '9' in second part (space-separated)
# -----------------------------------------------------------------------------
# SQL (split ngram by space, first part has '!', second part has '9'):
# Option A - element_at uses 1-based indexing:
spark.sql("""
    SELECT * FROM ngram_table
    WHERE ELEMENT_AT(SPLIT(ngram, ' '), 1) LIKE '%!%'
      AND ELEMENT_AT(SPLIT(ngram, ' '), 2) LIKE '%9%'
""").show()
# API:
df.filter(
    F.split(F.col("ngram"), " ").getItem(0).contains("!") &
    F.split(F.col("ngram"), " ").getItem(1).contains("9")
).show()

+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1985|    1|    1|    1|
|! $17.95|1987|    1|    1|    1|
|! $17.95|1990|    1|    1|    1|
|! $17.95|1991|    1|    1|    1|
|! $17.95|1992|    5|    5|    5|
|! $17.95|1993|    2|    2|    2|
|! $17.95|1995|    1|    1|    1|
|! $17.95|1996|    4|    2|    2|
|! $17.95|1997|    6|    5|    5|
|! $17.95|1998|    4|    3|    3|
|! $17.95|1999|   11|   10|   10|
|! $17.95|2000|   11|    9|    9|
|! $17.95|2001|    5|    4|    4|
|! $17.95|2002|    5|    5|    5|
|! $17.95|2003|    2|    2|    2|
|! $17.95|2004|   14|   14|   14|
|! $17.95|2005|   13|   13|   13|
|! $17.95|2006|    5|    5|    5|
|! $17.95|2007|    2|    2|    2|
|! $17.95|2008|    2|    2|    2|
+--------+----+-----+-----+-----+
only showing top 20 rows
+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1985|    1|    1|    1|
|! $17.95|1987|    1|  

In [ ]:
# 3.6) Bigrams that appeared in ALL years present in the dataset
# -----------------------------------------------------------------------------
# SQL:
spark.sql("""
    WITH total_years AS (SELECT COUNT(DISTINCT Year) as cnt FROM ngram_table),
         bigram_years AS (
             SELECT ngram, COUNT(DISTINCT Year) as years_count
             FROM ngram_table GROUP BY ngram
         )
    SELECT b.ngram FROM bigram_years b, total_years t
    WHERE b.years_count = t.cnt
""").show()
# API:
total_years = df.select(F.countDistinct("Year")).collect()[0][0]
df.groupBy("ngram") \
    .agg(F.countDistinct("Year").alias("years_count")) \
    .filter(F.col("years_count") == total_years) \
    .select("ngram") \
    .show()

+-----+
|ngram|
+-----+
| ! 09|
+-----+

+-----+
|ngram|
+-----+
| ! 09|
+-----+



In [ ]:
# 3.7) Total Pages and Books per bigram per year, sorted alphabetically
# -----------------------------------------------------------------------------
# SQL:
spark.sql("""
    SELECT ngram, Year, SUM(Pages) as total_pages, SUM(Books) as total_books
    FROM ngram_table
    GROUP BY ngram, Year
    ORDER BY ngram, Year
""").show()
# API:
df.groupBy("ngram", "Year") \
    .agg(F.sum("Pages").alias("total_pages"), F.sum("Books").alias("total_books")) \
    .orderBy("ngram", "Year") \
    .show()

+--------+----+-----------+-----------+
|   ngram|Year|total_pages|total_books|
+--------+----+-----------+-----------+
|! $17.95|1985|          1|          1|
|! $17.95|1987|          1|          1|
|! $17.95|1990|          1|          1|
|! $17.95|1991|          1|          1|
|! $17.95|1992|          5|          5|
|! $17.95|1993|          2|          2|
|! $17.95|1995|          1|          1|
|! $17.95|1996|          2|          2|
|! $17.95|1997|          5|          5|
|! $17.95|1998|          3|          3|
|! $17.95|1999|         10|         10|
|! $17.95|2000|          9|          9|
|! $17.95|2001|          4|          4|
|! $17.95|2002|          5|          5|
|! $17.95|2003|          2|          2|
|! $17.95|2004|         14|         14|
|! $17.95|2005|         13|         13|
|! $17.95|2006|          5|          5|
|! $17.95|2007|          2|          2|
|! $17.95|2008|          2|          2|
+--------+----+-----------+-----------+
only showing top 20 rows
+--------+----+

In [ ]:
# 3.8) Total distinct bigrams per year, sorted by year descending
# -----------------------------------------------------------------------------
# SQL:
spark.sql("""
    SELECT Year, COUNT(DISTINCT ngram) as distinct_bigrams
    FROM ngram_table
    GROUP BY Year
    ORDER BY Year DESC
""").show()
# API:
df.groupBy("Year").agg(F.countDistinct("ngram").alias("distinct_bigrams")) \
    .orderBy(F.col("Year").desc()) \
    .show()

+----+----------------+
|Year|distinct_bigrams|
+----+----------------+
|2008|               2|
|2007|               2|
|2006|               2|
|2005|               2|
|2004|               2|
|2003|               2|
|2002|               2|
|2001|               2|
|2000|               2|
|1999|               2|
|1998|               2|
|1997|               2|
|1996|               2|
|1995|               2|
|1993|               2|
|1992|               2|
|1991|               2|
|1990|               2|
|1987|               2|
|1985|               2|
+----+----------------+
only showing top 20 rows
+----+----------------+
|Year|distinct_bigrams|
+----+----------------+
|2008|               2|
|2007|               2|
|2006|               2|
|2005|               2|
|2004|               2|
|2003|               2|
|2002|               2|
|2001|               2|
|2000|               2|
|1999|               2|
|1998|               2|
|1997|               2|
|1996|               2|
|1995|         